# End-to-End Scikit-HEP Analysis Pipeline

In the previous chapters, you learned how to use individual libraries like `uproot`, `awkward`, `vector`, and `hist` in isolation. 

In this chapter, we will string them all together into a realistic, end-to-end physics analysis pipeline! Our goal is to "rediscover" the Higgs boson in the $H \rightarrow ZZ \rightarrow 4\ell$ channel using CERN Open Data.

## 1. Data Ingestion (`uproot`)
We begin by downloading our raw data. For this tutorial, we use a dataset from `skhep_testdata` which contains simulated CMS collision events.

In [ ]:
import uproot
import skhep_testdata
import awkward as ak
import vector
import hist
import matplotlib.pyplot as plt
import numpy as np

# Load the HZZ test file
filename = skhep_testdata.data_path("uproot-HZZ.root")
file = uproot.open(filename)
tree = file["events"]

# Print available branches
print(tree.keys())

## 2. Reading Data into Awkward Arrays
To search for $H \rightarrow ZZ \rightarrow 4\mu$, we need the kinematics of the muons in the event. We extract the 4-momentum components as jagged `awkward` arrays.

In [ ]:
muons = tree.arrays(["Muon_Px", "Muon_Py", "Muon_Pz", "Muon_E", "Muon_Charge"])
print(f"Total events loaded: {len(muons)}")
print("Muon energies for the first 5 events:")
print(muons.Muon_E[:5])

## 3. Filtering and Event Selection (`awkward`)
We are specifically looking for the $4\mu$ final state. Therefore, we filter our dataset to only keep events that contain exactly 4 muons.

In [ ]:
# Count muons per event
num_muons = ak.num(muons.Muon_E)

# Filter the arrays
mask_4mu = (num_muons == 4)
events_4mu = muons[mask_4mu]

print(f"Events remaining with exactly 4 muons: {len(events_4mu)}")

To reconstruct the Z bosons, we need to ensure charge conservation. A Z boson decays into a muon and an anti-muon (total charge 0). Thus, the sum of the charges of the 4 muons must be zero.

In [ ]:
# Ensure the sum of charges in the event is exactly 0
charge_sum = ak.sum(events_4mu.Muon_Charge, axis=1)
mask_charge = (charge_sum == 0)

final_events = events_4mu[mask_charge]
print(f"Events passing charge conservation: {len(final_events)}")

## 4. Kinematics (`vector`)
We now construct Lorentz 4-vectors to calculate the invariant mass of the four muons. Since all 4 muons come from the original Higgs boson (via two Z bosons), summing their 4-momenta yields the 4-momentum of the Higgs!

In [ ]:
# Construct the 4-vector array
vector.register_awkward()

muon_p4 = ak.zip({
    "px": final_events.Muon_Px,
    "py": final_events.Muon_Py,
    "pz": final_events.Muon_Pz,
    "E": final_events.Muon_E
}, with_name="Momentum4D")

# Sum the 4 muons in each event
higgs_p4 = ak.sum(muon_p4, axis=1)

# Extract the invariant mass
higgs_mass = higgs_p4.mass
print("Reconstructed Higgs Masses (GeV):")
print(higgs_mass[:10])

## 5. Visualization (`hist`)
Finally, we bin the data and plot it. In a real dataset with much more statistics, you would see a peak at $125\text{ GeV}$ representing the Higgs boson, and another peak near $91\text{ GeV}$ representing single Z bosons where a radiated photon or misidentified track faked the other muons.

In [ ]:
# Create a histogram from 70 to 180 GeV
h = hist.Hist.new.Reg(30, 70, 180, name="mass", label="$m_{4\mu}$ [GeV]").Double()
h.fill(higgs_mass)

# Plotting
fig, ax = plt.subplots(figsize=(8, 6))
h.plot(ax=ax, histtype="fill", color="steelblue", edgecolor="black")
ax.set_title("Invariant Mass of $4\mu$ System")
ax.set_ylabel("Events / Bin")
plt.show()

**Congratulations!** You have just executed a full HEP analysis pipeline purely in Python, transforming raw `.root` trees into jagged arrays, slicing out physics events, calculating Lorentz invariants, and extracting the physics!